# 02. Distribution fitting and design rainfall

This notebook fits four candidate distributions to the observed annual maximum series, compares them with three goodness-of-fit statistics and derives the 24-h design rainfall depths for the 2-, 5-, 10-, 25-, 50- and 100-year return periods. It produces the goodness-of-fit table, the observed column of the design rainfall table and the figure of fitted distributions.


## Configuration

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
%matplotlib inline

ROOT = Path("..")
INPUT_AMS = ROOT / "results" / "01_trend" / "AMS.csv"
OUTDIR = ROOT / "results" / "02_frequency"
RETURN_PERIODS = [2, 5, 10, 25, 50, 100]

OUTDIR.mkdir(parents=True, exist_ok=True)
ams = pd.read_csv(INPUT_AMS)
x = ams["Rainfall_mm"].values
print(f"{len(x)} annual maxima, {int(ams['Year'].min())} to {int(ams['Year'].max())}")

## 1. Sample L-moments

The sample L-moments $\lambda_1$, $\lambda_2$ and the L-skewness $\tau_3 = \lambda_3/\lambda_2$ are computed from the probability-weighted moments of the ordered sample (Hosking 1990).

In [ ]:
def lmoments(x):
    """Sample L-moments l1, l2 and L-skewness t3 from probability-weighted moments."""
    x = np.sort(np.asarray(x, float))
    n = len(x)
    j = np.arange(1, n + 1)
    b0 = np.mean(x)
    b1 = np.sum((j - 1) / (n - 1) * x) / n
    b2 = np.sum((j - 1) * (j - 2) / ((n - 1) * (n - 2)) * x) / n
    l1 = b0
    l2 = 2 * b1 - b0
    t3 = (6 * b2 - 6 * b1 + b0) / l2
    return l1, l2, t3

## 2. Parameter estimation

- **Gumbel (EV1):** method of moments, scale $\alpha_G = s\sqrt{6}/\pi$ and location $u = \bar{x} - 0.5772\,\alpha_G$.
- **Generalised Extreme Value (GEV):** L-moments, with the approximation of Hosking et al. (1985),
  $c = \dfrac{2}{3+\tau_3} - \dfrac{\ln 2}{\ln 3}$, $\kappa = 7.8590\,c + 2.9554\,c^2$,
  $\sigma = \dfrac{\lambda_2\,\kappa}{(1-2^{-\kappa})\,\Gamma(1+\kappa)}$, $\mu = \lambda_1 - \dfrac{\sigma\,[1-\Gamma(1+\kappa)]}{\kappa}$.
  The shape argument of `scipy.stats.genextreme` equals $\kappa$ in this parameterisation.
- **Log-Pearson Type III:** moments of the base-10 logarithms, with the bias-corrected sample skewness.
- **Lognormal (two-parameter):** mean and standard deviation of the natural logarithms.

The design depth for return period $T$ is the quantile at non-exceedance probability $1 - 1/T$.

In [ ]:
def fit_gumbel(x):
    s, m = np.std(x, ddof=1), np.mean(x)
    scale = s * math.sqrt(6) / math.pi
    loc = m - 0.5772 * scale
    return {"name": "Gumbel", "dist": stats.gumbel_r,
            "kw": {"loc": loc, "scale": scale},
            "report": f"loc={loc:.2f}, scale={scale:.2f}"}

def fit_gev(x):
    l1, l2, t3 = lmoments(x)
    cc = 2.0 / (3.0 + t3) - math.log(2) / math.log(3)
    k = 7.8590 * cc + 2.9554 * cc ** 2
    k = k if abs(k) > 1e-6 else 1e-6
    gk = math.gamma(1 + k)
    scale = l2 * k / ((1 - 2 ** (-k)) * gk)
    loc = l1 - scale * (1 - gk) / k
    return {"name": "GEV", "dist": stats.genextreme,
            "kw": {"c": k, "loc": loc, "scale": scale},
            "report": f"shape={k:.4f}, loc={loc:.2f}, scale={scale:.2f}"}

def fit_lp3(x):
    y = np.log10(x)
    m, s = np.mean(y), np.std(y, ddof=1)
    sk = stats.skew(y, bias=False)
    sk = sk if abs(sk) > 1e-4 else 1e-4
    return {"name": "Log-Pearson III", "dist": stats.pearson3,
            "kw": {"skew": sk, "loc": m, "scale": s}, "log10": True,
            "report": f"log-skew={sk:.4f}, log-mean={m:.4f}, log-sd={s:.4f}"}

def fit_lognormal(x):
    y = np.log(x)
    m, s = np.mean(y), np.std(y, ddof=1)
    return {"name": "Lognormal", "dist": stats.norm,
            "kw": {"loc": m, "scale": s}, "ln": True,
            "report": f"ln-mean={m:.4f}, ln-sd={s:.4f}"}

# transforms for the two distributions fitted in log space
def _fwd(m, x):  return np.log10(x) if m.get("log10") else (np.log(x) if m.get("ln") else x)
def _back(m, y): return 10 ** y     if m.get("log10") else (np.exp(y) if m.get("ln") else y)
def cdf(m, x):   return m["dist"].cdf(_fwd(m, x), **m["kw"])
def quantile(m, T):
    p = 1 - 1.0 / T
    return float(_back(m, m["dist"].ppf(p, **m["kw"])))

models = [fit_gumbel(x), fit_gev(x), fit_lp3(x), fit_lognormal(x)]
for m in models:
    print(f"{m['name']:<18s} {m['report']}")

## 3. Goodness of fit

Three statistics are computed for each fitted distribution $F$:

- the Kolmogorov-Smirnov statistic $D = \max_x |F_n(x) - F(x)|$, compared with the critical value $1.36/\sqrt{n}$ at a significance level of 0.05 (Massey 1951);
- the Anderson-Darling statistic (Anderson and Darling 1954), $A^2 = -n - \frac{1}{n}\sum_{i=1}^{n}(2i-1)\left[\ln F(x_{(i)}) + \ln\{1 - F(x_{(n+1-i)})\}\right]$;
- the chi-square statistic on eight equiprobable classes of the fitted distribution.

The distributions are ranked on each statistic, with tied ranks averaged, and the distribution with the lowest mean rank is adopted.

In [ ]:
def ks_stat(m, x):
    xs = np.sort(x); n = len(xs)
    F = cdf(m, xs)
    d = max(np.max(np.abs(F - (np.arange(1, n + 1) - 1) / n)),
            np.max(np.abs(F - np.arange(1, n + 1) / n)))
    crit = 1.36 / math.sqrt(n)
    return float(d), float(crit)

def ad_stat(m, x):
    xs = np.sort(x); n = len(xs)
    F = np.clip(cdf(m, xs), 1e-12, 1 - 1e-12)
    i = np.arange(1, n + 1)
    return float(-n - np.sum((2 * i - 1) * (np.log(F) + np.log(1 - F[::-1]))) / n)

def chi2_stat(m, x, nbins=8):
    n = len(x)
    edges_p = np.linspace(0, 1, nbins + 1)
    edges = [_back(m, m["dist"].ppf(p, **m["kw"])) for p in edges_p[1:-1]]
    edges = [-np.inf] + list(edges) + [np.inf]
    obs, _ = np.histogram(x, bins=edges)
    exp = np.full(nbins, n / nbins)
    return float(np.sum((obs - exp) ** 2 / exp))

rows = []
for m in models:
    d, crit = ks_stat(m, x)
    rows.append({
        "Distribution": m["name"], "Parameters": m["report"],
        "K-S": round(d, 4), "K-S critical": round(crit, 4),
        "K-S verdict": "accept" if d < crit else "reject",
        "Anderson-Darling": round(ad_stat(m, x), 4),
        "Chi-square": round(chi2_stat(m, x), 4),
    })
gof = pd.DataFrame(rows)
gof["r1"] = gof["K-S"].rank()
gof["r2"] = gof["Anderson-Darling"].rank()
gof["r3"] = gof["Chi-square"].rank()
gof["Mean rank"] = gof[["r1", "r2", "r3"]].mean(axis=1).round(2)
gof = gof.sort_values("Mean rank").reset_index(drop=True)
gof["Selected"] = ["selected" if i == 0 else "" for i in range(len(gof))]

gof_show = gof.drop(columns=["r1", "r2", "r3"])
gof_show.to_csv(OUTDIR / "goodness_of_fit.csv", index=False)
gof_show

In [ ]:
CHOSEN = gof.loc[0, "Distribution"]
best = next(m for m in models if m["name"] == CHOSEN)
print(f"Selected distribution: {best['name']}")
print(f"Parameters: {best['report']}")

## 4. Fitted distributions and observations

The observations are plotted at the Hazen position, which assigns the event of descending rank $m$ among $n$ values the return period $T_m = 2n/(2m - 1)$ (Cunnane 1978). Tied values receive their average rank.

In [ ]:
xs = np.sort(x)[::-1]
n = len(xs)
ranks = stats.rankdata(-xs, method="average")   # descending rank; ties share their average rank
hazen_T = 2 * n / (2 * ranks - 1)

fig, ax = plt.subplots(figsize=(8, 5))
Tfine = np.logspace(np.log10(1.05), np.log10(200), 200)
for m in models:
    is_best = (m["name"] == CHOSEN)
    ax.plot(Tfine, [quantile(m, t) for t in Tfine],
            lw=2.4 if is_best else 1.1,
            ls="-" if is_best else "--",
            label=m["name"] + (" (selected)" if is_best else ""))
ax.scatter(hazen_T, xs, s=26, c="k", zorder=5, label="Observed (Hazen)")
ax.set_xscale("log")
ax.set_xlabel("Return period (years)")
ax.set_ylabel("Annual maximum daily rainfall (mm)")
ax.set_title("Comparison of Probability Distributions")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(OUTDIR / "fig_distribution_comparison.png", dpi=300)
plt.show()

## 5. Design rainfall depths

Quantiles of all four distributions are tabulated for comparison. The column of the selected distribution holds the present-climate 24-h design depths that are carried forward.

In [ ]:
rows = []
for T in RETURN_PERIODS:
    row = {"Return period (yr)": T, "Exceedance prob (%)": round(100.0 / T, 2)}
    for m in models:
        row[f"{m['name']} (mm)"] = round(quantile(m, T), 1)
    rows.append(row)

design = pd.DataFrame(rows)
design.to_csv(OUTDIR / "design_rainfall_observed.csv", index=False)
print(f"Design depths: column '{CHOSEN} (mm)'")
design